<a href="https://colab.research.google.com/github/marwa-nassane0052/quantum_ai_experiment/blob/main/test8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

folder_path = '/content/drive/MyDrive/skin_cancer_processed_dataset_data_augmentation'

if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Folder '{folder_path}' not found. Please check the path and ensure it's in your My Drive.")

Contents of '/content/drive/MyDrive/skin_cancer_processed_dataset_data_augmentation':
y_train.npy
y_test.npy
X_train.npy
X_test.npy


In [4]:
import numpy as np
folder_path = '/content/drive/MyDrive/skin_cancer_processed_dataset_data_augmentation'
X_train = np.load(os.path.join(folder_path, 'X_train.npy'))
y_train = np.load(os.path.join(folder_path, 'y_train.npy'))
X_test = np.load(os.path.join(folder_path, 'X_test.npy'))
y_test = np.load(os.path.join(folder_path, 'y_test.npy'))

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2200, 3, 128, 128)
y_train shape: (2200,)
X_test shape: (660, 3, 128, 128)
y_test shape: (660,)


**Bulding the hybrid model**

In [3]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 98.3 MB/s eta 0:00:00


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# ── Convert dataset to tensors ─────────────────────────────────────────────────
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=16)

# ── Quantum setup ──────────────────────────────────────────────────────────────
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(2)]

weight_shapes = {"weights": (3, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qlayer

    def forward(self, x):
        return torch.stack([self.qlayer(x[i]) for i in range(x.shape[0])])

# ── Vision Transformer Components ─────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    """Split image into patches and linearly embed them."""
    def __init__(self, img_size=128, patch_size=16, in_channels=3, embed_dim=128):
        super().__init__()
        assert img_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size  = patch_size
        # Conv2d with kernel=patch_size & stride=patch_size acts as a patch splitter + linear projection
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W) → (B, embed_dim, n_patches_h, n_patches_w) → (B, num_patches, embed_dim)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class TransformerEncoderBlock(nn.Module):
    """Single ViT encoder block: Multi-Head Self-Attention + MLP."""
    def __init__(self, embed_dim=128, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Self-attention with residual
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        # MLP with residual
        x = x + self.mlp(self.norm2(x))
        return x


class ViTFeatureExtractor(nn.Module):
    """
    Lightweight Vision Transformer that outputs a fixed-size feature vector.
    img_size   : height (= width) of input images  → adjust to your data
    patch_size : size of each square patch          → 16 works well for 128×128
    embed_dim  : token embedding dimension
    depth      : number of transformer encoder blocks
    num_heads  : attention heads (must divide embed_dim)
    out_dim    : size of the final feature vector fed to the quantum layer
    """
    def __init__(self, img_size=128, patch_size=16, in_channels=3,
                 embed_dim=128, depth=4, num_heads=4, out_dim=n_qubits, dropout=0.1):
        super().__init__()

        # 1. Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        # 2. [CLS] token + positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        # 3. Transformer encoder blocks
        self.blocks = nn.Sequential(*[
            TransformerEncoderBlock(embed_dim, num_heads, dropout=dropout)
            for _ in range(depth)
        ])

        # 4. Final norm + projection → n_qubits features
        self.norm    = nn.LayerNorm(embed_dim)
        self.proj_out = nn.Linear(embed_dim, out_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]

        # Patch + position embedding
        x = self.patch_embed(x)                                     # (B, num_patches, embed_dim)
        cls = self.cls_token.expand(B, -1, -1)                      # (B, 1, embed_dim)
        x   = torch.cat([cls, x], dim=1)                            # (B, num_patches+1, embed_dim)
        x   = self.pos_drop(x + self.pos_embed)

        # Transformer blocks
        x = self.blocks(x)                                          # (B, num_patches+1, embed_dim)

        # Use [CLS] token representation
        cls_out = self.norm(x[:, 0])                                # (B, embed_dim)

        # Project to qubit-compatible size
        out = self.proj_out(cls_out)                                # (B, n_qubits)
        return torch.tanh(out)                                      # scale to [-1, 1] for AngleEmbedding


# ── Hybrid ViT + Quantum Model ─────────────────────────────────────────────────
class HybridViTQuantumModel(nn.Module):
    """
    Full hybrid model:
      ViT feature extractor  →  linear projection  →  Quantum classifier
    """
    def __init__(self, img_size=128):
        super().__init__()
        self.vit     = ViTFeatureExtractor(img_size=img_size)   # outputs (B, n_qubits)
        self.quantum = QuantumLayer()                            # outputs (B, 2)

    def forward(self, x):
        features = self.vit(x)       # (B, n_qubits), already tanh-scaled
        out      = self.quantum(features)  # (B, 2)
        return out


# ── Detect image size from data ────────────────────────────────────────────────
# X_train shape is expected to be (N, C, H, W)
img_size = X_train_t.shape[-1]   # e.g. 128 for 128×128 images
print(f"Detected image size: {img_size}×{img_size}")

model     = HybridViTQuantumModel(img_size=img_size)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# optional: learning-rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# ── Training loop ──────────────────────────────────────────────────────────────
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = torch.max(outputs, 1)
        total        += labels.size(0)
        correct      += (predicted == labels).sum().item()

    scheduler.step()
    train_acc  = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

# ── Evaluation ─────────────────────────────────────────────────────────────────
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs     = model(images)
        _, predicted = torch.max(outputs, 1)
        total       += labels.size(0)
        correct     += (predicted == labels).sum().item()

print(f"\nTest Accuracy: {100 * correct / total:.2f}%")

Detected image size: 128×128
Epoch 01/20 | Loss: 0.6330 | Train Acc: 68.68%
Epoch 02/20 | Loss: 0.5858 | Train Acc: 75.95%
Epoch 03/20 | Loss: 0.5730 | Train Acc: 76.59%
Epoch 04/20 | Loss: 0.5577 | Train Acc: 77.09%
Epoch 05/20 | Loss: 0.5336 | Train Acc: 79.64%
Epoch 06/20 | Loss: 0.5232 | Train Acc: 80.36%
Epoch 07/20 | Loss: 0.5131 | Train Acc: 81.50%
Epoch 08/20 | Loss: 0.5041 | Train Acc: 82.36%
Epoch 09/20 | Loss: 0.5016 | Train Acc: 82.36%
Epoch 10/20 | Loss: 0.4936 | Train Acc: 83.27%
Epoch 11/20 | Loss: 0.4898 | Train Acc: 83.64%
Epoch 12/20 | Loss: 0.4914 | Train Acc: 83.64%
Epoch 13/20 | Loss: 0.4955 | Train Acc: 82.91%
Epoch 14/20 | Loss: 0.4938 | Train Acc: 83.55%
Epoch 15/20 | Loss: 0.5039 | Train Acc: 82.09%
Epoch 16/20 | Loss: 0.4959 | Train Acc: 81.86%
Epoch 17/20 | Loss: 0.5007 | Train Acc: 81.41%
Epoch 18/20 | Loss: 0.5034 | Train Acc: 81.09%
Epoch 19/20 | Loss: 0.5038 | Train Acc: 80.27%
Epoch 20/20 | Loss: 0.4918 | Train Acc: 80.23%

Test Accuracy: 78.64%
